Double-entry bookkeeping records every business transaction as an *entry* that
debits one account and credits another. In this chapter we build a small ledger in Python
and, in doing so, revisit the whole progression of this course:

1. **A data-carrying class** that models a single booking (`Entry`).
2. **Bundling** state and behaviour into a class that keeps a running balance (`Account`).
3. **Inheritance** — the new idea: a current account *is an* `Account`, so it can inherit
   everything the `Account` class already knows and add only what interest-bearing accounts need.

::: {.callout-note icon=false}
## Bookkeeping Fundamentals

Every entry touches two accounts. By convention an account's balance (its *Saldo*) moves
according to which side the entry lands on:

- a **debit** on the account increases its balance;
- a **credit** on the account decreases it.

The *interest* on a balance held over a period follows the simple day-count formula

$$\text{interest} = \text{balance} \times \frac{\text{rate}}{360} \times \text{days},$$

where the year is taken as 360 days (the *30/360* convention common in Swiss practice).
:::


## The Design at a Glance

Before writing any code, it helps to picture where we are heading. @fig-uml is a **UML class
diagram** of the three classes in this chapter and how they relate.


UML class diagram




::: {.content-visible when-format="html"}

```{mermaid}
%%| label: fig-uml
%%| fig-cap: "UML class diagram: CurrentAccount inherits from Account; each Account holds Entry objects."
%%| echo: false
classDiagram
    class Entry {
        +int debit
        +int credit
        +float amount
        +date date
        +dict accounting_scheme
        +__str__() str
    }
    class Account {
        +str name
        +int accountNr
        +float startAmount
        +list~Entry~ entries
        +float saldo
        +add_entry(entry) None
        +calculate_saldo() float
    }
    class CurrentAccount {
        +float interest_rate
        +list saldi
        +add_entry(entry) None
        +calculate_interest(end_date) float
    }
    Account <|-- CurrentAccount : is-a (inherits)
    Account o-- Entry : has-many
```

:::

::: {.content-hidden when-format="html"}

![UML class diagram: CurrentAccount inherits from Account; each Account holds Entry objects.](uml-class-diagram.png){#fig-uml}

:::

Two relationships appear here, and telling them apart is the whole point of the chapter:

- The **hollow triangle** arrow (`Account <|-- CurrentAccount`) is *inheritance*: a
  `CurrentAccount` **is an** `Account`. It inherits every attribute and method of the parent
  and adds `interest_rate`, `saldi` and `calculate_interest`. Note that `add_entry` appears in
  *both* boxes — the subclass **overrides** it.
- The **hollow diamond** arrow (`Account o-- Entry`) is *aggregation*: an `Account` **has**
  a list of `Entry` objects. This is composition, not inheritance — an account is not a kind
  of entry.

::: {.callout-note icon=false}
## Reading a UML Class Diagram

Each box has three compartments: the class **name**, its **attributes** (data) and its
**methods** (behaviour). A `+` marks a public member. The arrow *type* encodes the
relationship — an *is-a* link (inheritance) uses a hollow triangle pointing at the parent,
whereas a *has-a* link (aggregation) uses a hollow diamond at the containing class.
:::


## Stage 1 — A Class for a Single Entry {#sec-inh-stage1}

We begin with the smallest unit of the ledger: one booking. An `Entry` stores the debit
account number, the credit account number, an amount and a date. It also carries the chart
of accounts (*Kontenrahmen*) so that it can print itself in a human-readable form.


::: {.callout-note icon=false}
## Dunder Methods

Methods whose names begin and end with two underscores — `__init__`, `__str__`, `__len__`,
`__eq__` — are called **dunder** methods (from *double underscore*). They are also known as
*magic* or *special* methods. The double underscores mark them as part of Python's own
protocol, so they never clash with method names you choose yourself.

You rarely call a dunder method by name. Instead, Python invokes it for you when you use
ordinary syntax or a built-in function: creating an `Entry` runs `__init__`; `print(entry)`
and `str(entry)` run `__str__`; `len(x)` runs `__len__`; and `x == y` runs `__eq__`. (We do
not need `__len__` or `__eq__` here — they only show the pattern.)

Defining a dunder method is how a class connects to that syntax and lets its objects behave
like built-in types. In this chapter we implement two. `__init__` sets a new object up.
`__str__` must return a string; defining it lets an `Entry` describe itself — date, both
account names and the amount — instead of showing the default `<Entry object at 0x…>`.
:::


::: {.callout-note icon=false}
## What `self.` Does

A class bundles data and the methods that act on it. The class definition is only a
*template*; from it, individual objects are created at run time, and each object carries its
own set of attribute values.

The first parameter of every method refers to the object the method was called on. It is
named `self` by convention, not by language rule — but always write `self`. When Python runs
`first_entry = Entry(1000, 1020, 50.00, date(2026, 8, 28))`, it runs `__init__` with `self`
bound to the new object. Later, `print(first_entry)` calls `first_entry.__str__()` with
`self` bound to that same object.

So inside `__init__`, `self.debit = debit` stores the value *on that one object*, and inside
`__str__`, `self.debit` and `self.accounting_scheme` read the data back from *the same*
object. Any value the object should still have after the method finishes must be stored on
`self`. A plain local name like `text` in `__str__` is gone as soon as the method returns.
:::


In [ ]:
from datetime import date

In [ ]:
class Entry:
    """A single double-entry booking (one debit, one credit).

    Attributes:
        debit (int): Account number debited.
        credit (int): Account number credited.
        amount (float): The amount booked.
        date (date): The date of the booking.
    """

    def __init__(self, debit: int,
                 credit: int,
                 amount: float,
                 date: date) -> None:
        self.debit = debit
        self.credit = credit
        self.amount = amount
        self.date = date
        # Abridged chart of accounts (Kontenrahmen KMU) for readable output.
        self.accounting_scheme = {
            1000: "Kasse",
            1020: "Bankguthaben",
            2000: "Verbindlichkeiten aus Lieferungen und Leistungen (Kreditoren)",
            6950: "Finanzertrag",
        }

    def __str__(self) -> str:
        text = (
            f"{self.date.strftime('%d %b %Y')}: "
            f"{self.accounting_scheme[self.debit]} "
            f"({self.debit}) - {self.accounting_scheme[self.credit]} "
            f"({self.credit}) {self.amount}"
        )
        return " ".join(text.split())

A first booking: 50.00 moved from *Bankguthaben* (1020) to *Kasse* (1000). Printing the
object triggers `__str__`.

In [ ]:
first_entry = Entry(1000, 1020, 50.00, date(2026, 8, 28))
print(first_entry)

## Stage 2 — Bundling into an Account {#sec-inh-stage2}

A single entry is of little use on its own. An **account** collects the entries that touch
it and can compute its balance at any time. We bundle that state (name, number, opening
balance, list of entries) and the behaviour (`add_entry`, `calculate_saldo`) into one class.

`calculate_saldo` walks through every entry: an entry that **debits** this account raises
the balance, one that **credits** it lowers the balance, and entries touching neither are
skipped.


In [ ]:
class Account:
    """A ledger account that collects entries and computes its balance.

    Attributes:
        name (str): Human-readable account name.
        accountNr (int): The account number.
        startAmount (float): Opening balance.
        entries (list[Entry]): All entries booked to this account.
        saldo (float | None): Most recently computed balance.
    """

    def __init__(self,
                 name: str,
                 accountNr: int,
                 startAmount: float) -> None:
        self.name = name
        self.accountNr = accountNr
        self.startAmount = startAmount
        self.entries = []
        self.saldo = None

    def add_entry(self, entry: Entry) -> None:
        """Appends an entry to the account and reports it.

        Args:
            entry (Entry): The booking to add.
        """
        self.entries.append(entry)
        print(f'Entry "{entry}" added.')

    def calculate_saldo(self) -> float:
        """Computes the balance from the opening amount and all entries.

        A debit on this account increases the balance; a credit
        decreases it.

        Returns:
            float: The resulting balance.
        """
        saldo = self.startAmount
        for entry in self.entries:
            if entry.debit == self.accountNr:
                saldo += entry.amount
            elif entry.credit == self.accountNr:
                saldo -= entry.amount
            else:
                continue
        print(f'Saldo: {saldo}')
        self.saldo = saldo
        return saldo

We open the cash account 1000 with 10.00, book two entries and read off the balance.

In [ ]:
a1000 = Account('Cash', 1000, 10.00)
a1000.add_entry(Entry(1000, 1020, 50.0, date(2026, 8, 28)))
a1000.add_entry(Entry(2000, 1000, 5.00, date(2026, 8, 28)))
a1000.calculate_saldo()

## Stage 3 — Inheritance: a Current Account *is an* Account {#sec-inh-stage3}

A **current account** (*Kontokorrent*) behaves exactly like an ordinary account — it collects
entries and has a balance — but it does something extra: it earns **interest**. That is the
textbook signal for inheritance: a current account *is an* account, plus a little more. In
the UML diagram (@fig-uml) this is the hollow-triangle link from `CurrentAccount` to `Account`.

Rather than copying all of `Account`, we let a new class **inherit** from it and add only
what interest requires.

::: {.callout-note icon=false}
## The `is-a` Relationship and `super()`

Writing `class CurrentAccount(Account)` makes `CurrentAccount` a **subclass** (child) of
`Account`. It automatically gains every method of the parent — `add_entry`, `calculate_saldo`
and the constructor. Inside its own `__init__`, the call `super().__init__(...)` runs the
parent's constructor, so `name`, `accountNr`, `startAmount`, `entries` and `saldo` are set up
for us. We then add just the new attributes `interest_rate` and `saldi`.

Contrast this with *composition* (the hollow-diamond `has-a` link between `Account` and
`Entry`): inheritance is the right tool when the new type genuinely **is a** specialised kind
of the old one.
:::


### A First Version {#sec-first-version}

We build the subclass step by step. The constructor takes one extra argument, `interest_rate`,
hands the rest to the parent via `super().__init__(...)`, and keeps a dated history of balances
(`saldi`) so that interest can later be computed period by period. `add_entry` is **overridden**:
it first calls `super().add_entry(entry)` to keep the inherited behaviour, then records the new
balance together with the entry's date.

For `calculate_interest` we write the calculation out in full, in the most direct way we can
think of — index by index. It works; we will scrutinise it afterwards.


In [ ]:
class CurrentAccount(Account):
    """An interest-bearing account — first version.

    Inherits entry handling and balance calculation from Account and
    adds an interest rate together with a dated history of balances.

    Attributes:
        interest_rate (float): Annual interest rate (e.g. 0.05 for 5%).
        saldi (list[tuple[float, date]]): Balance history as
            (balance, date) pairs, opened with the start amount.
    """

    def __init__(self,
                 name: str,
                 accountNr: int,
                 startAmount: float,
                 interest_rate: float) -> None:
        super().__init__(name, accountNr, startAmount)  # parent sets up the account
        self.interest_rate = interest_rate              # the new attributes
        self.saldi = []
        self.saldi.append((startAmount, date(2026, 1, 1)))

    def add_entry(self, entry: Entry) -> None:
        """Extends Account.add_entry by recording the new dated balance."""
        super().add_entry(entry)                                 # inherited behaviour
        self.saldi.append((self.calculate_saldo(), entry.date))  # plus history

    def calculate_interest(self, end_date=date(2026, 12, 31)) -> float:
        """Computes 30/360 interest across the balance history.

        Each balance earns interest over the number of days it was held,
        i.e. until the next balance change (and finally until end_date).
        If end_date is the year-end, the total is booked as Finanzertrag.
        """
        number_of_days = []
        interests = []
        end_date = end_date
        i = 0
        saldi = [saldo[0] for saldo in self.saldi]
        day_list = [saldo[1] for saldo in self.saldi]
        while i < (len(day_list) - 1):
            d = (day_list[i+1] - day_list[i]).days
            number_of_days.append(d)
            i += 1
        rest_days = (end_date - day_list[-1]).days
        number_of_days.append(rest_days)
        for i in range(len(saldi)):
            interest = (saldi[i] * self.interest_rate / 360) * number_of_days[i]
            interests.append(interest)
        if end_date == date(2026, 12, 31):
            entry = Entry(1020, 6950, sum(interests), end_date)
            self.add_entry(entry)
        return sum(interests)

We open a bank current account 1020 with 100.00 at 5 % interest, book a movement in March,
and compute the interest to the year-end.

In [ ]:
bank = CurrentAccount('Bank', 1020, 100.00, 0.05)
bank.add_entry(Entry(1020, 1000, 150.00, date(2026, 3, 1)))
print(bank.calculate_interest())

### Refactoring `calculate_interest` {#sec-refactoring}

The first version is correct, but it does more bookkeeping of its own than the problem
requires. Reading it closely, a few things stand out:

- Two accumulator lists (`number_of_days`, `interests`) are filled with `append` inside loops.
- A manual index `i` and a `while` loop walk through consecutive dates — a classic place for
  off-by-one errors.
- The line `end_date = end_date` does nothing.

Python lets us say the same thing more directly. The days between consecutive balances are
just the differences of **adjacent pairs**, which `zip(day_list, day_list[1:])` produces
without any index. Both loops then collapse into **list comprehensions**, and the intent —
*"each balance, times its days, times the daily rate"* — becomes readable in one line.

::: {.callout-note icon=false}
## Pairing Adjacent Elements with `zip`

`zip(seq, seq[1:])` pairs each element with its successor: from `[d0, d1, d2]` it yields
`(d0, d1)` and `(d1, d2)`. Iterating over those pairs replaces the manual index arithmetic
`day_list[i+1] - day_list[i]` and removes the risk of mis-counting the loop bounds.
:::

The behaviour is identical — only `calculate_interest` changes; everything inherited from
`Account` stays untouched.


In [ ]:
class CurrentAccount(Account):
    """An interest-bearing account — refactored version.

    Identical behaviour to the first version; only calculate_interest
    is rewritten using zip and list comprehensions.
    """

    def __init__(self,
                 name: str,
                 accountNr: int,
                 startAmount: float,
                 interest_rate: float) -> None:
        super().__init__(name, accountNr, startAmount)
        self.interest_rate = interest_rate
        self.saldi = []
        self.saldi.append((startAmount, date(2026, 1, 1)))

    def add_entry(self, entry: Entry) -> None:
        """Extends Account.add_entry by recording the new dated balance."""
        super().add_entry(entry)
        self.saldi.append((self.calculate_saldo(), entry.date))

    def calculate_interest(self, end_date=date(2026, 12, 31)) -> float:
        """Computes 30/360 interest across the balance history.

        Args:
            end_date (date, optional): End of the interest period.
                Defaults to 31 December 2026.

        Returns:
            float: The total interest over the period.
        """
        saldi = [saldo[0] for saldo in self.saldi]
        day_list = [saldo[1] for saldo in self.saldi]

        # Days between consecutive balances (adjacent pairs via zip)...
        number_of_days = [
            (nxt - cur).days
            for cur, nxt in zip(day_list, day_list[1:])
        ]
        # ... plus the remaining days from the last balance to end_date.
        number_of_days.append((end_date - day_list[-1]).days)

        # Charge each balance interest over the days it was held.
        interests = [
            saldo * self.interest_rate / 360 * days
            for saldo, days in zip(saldi, number_of_days)
        ]

        if end_date == date(2026, 12, 31):
            entry = Entry(1020, 6950, sum(interests), end_date)
            self.add_entry(entry)
        return sum(interests)

Running the same example confirms that the refactored version yields exactly the same
interest — same result, clearer code.

In [ ]:
bank = CurrentAccount('Bank', 1020, 100.00, 0.05)
bank.add_entry(Entry(1020, 1000, 150.00, date(2026, 3, 1)))
print(bank.calculate_interest())

Because `CurrentAccount` *is an* `Account`, every inherited method just works — and
`isinstance()` confirms the relationship.

In [ ]:
print("Is an Account:", isinstance(bank, Account))
print("Balance:", bank.saldo)